In [ ]:
!pip install ultralytics
!pip install easyocr
!pip install supervision

In [ ]:
from google.colab import files
uploaded =files.upload()

In [ ]:
import os

os.listdir()

In [ ]:
import os

print(os.path.exists("output.mp4"))

In [ ]:
from IPython.display import Video

Video("output.mp4", embed=True)

In [ ]:
from IPython.display import Video

Video("tracked_output.mp4", embed=True)

In [ ]:
from IPython.display import Video

Video("tracked_output.mp4", embed=True)

In [ ]:
from IPython.display import Video

Video("tracked_output.mp4", embed=True)

In [ ]:
import os

size = os.path.getsize("tracked_output.mp4")

print("Size in MB:", size / (1024*1024))

In [ ]:
from google.colab import files

files.download("tracked_output.mp4")

In [ ]:
!wget -O license_plate_detector.pt https://github.com/ultralytics/assets/releases/download/v0.0.0/license_plate_detector.pt

In [ ]:
!rm -f license_plate_detector.pt

In [ ]:
!wget https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt

In [ ]:
!wget https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt

In [ ]:
import os

print(os.path.exists("plate.pt"))

In [ ]:
!wget -O plate.pt "https://github.com/keremberke/yolov8n-license-plate/releases/download/v1.0/model.pt"

In [ ]:
import os

print(os.path.exists("plate.pt"))

In [ ]:
import os

print(os.listdir())

In [ ]:
!rm -f plate.pt

In [ ]:
!wget -O plate.pt https://huggingface.co/keremberke/yolov8n-license-plate/resolve/main/best.pt

In [ ]:
import os

print(os.path.exists("plate.pt"))

In [ ]:
from ultralytics import YOLO
import cv2
import easyocr
import pandas as pd
from datetime import datetime

vehicle_model = YOLO("yolov8n.pt")

reader = easyocr.Reader(['en'])


video_path = "traffic.mp4"

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))


out = cv2.VideoWriter(
    "final_output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)


LINE_Y = 400


vehicle_positions = {}
crossed_ids = set()

records = []


while True:

    ret, frame = cap.read()

    if not ret:
        break

    # TRACK VEHICLES
    results = vehicle_model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml"
    )

    annotated_frame = results[0].plot()

    # DRAW LINE
    cv2.line(
        annotated_frame,
        (0, LINE_Y),
        (width, LINE_Y),
        (0, 255, 0),
        3
    )

    if results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()

        for box, track_id in zip(boxes, ids):

            x1, y1, x2, y2 = map(int, box)

            center_y = int((y1 + y2) / 2)

            previous_y = vehicle_positions.get(track_id, center_y)

            vehicle_positions[track_id] = center_y

            # LINE CROSSING
            if previous_y < LINE_Y and center_y >= LINE_Y:

                if track_id not in crossed_ids:

                    crossed_ids.add(track_id)

                    vehicle_crop = frame[y1:y2, x1:x2]

                    h, w, _ = vehicle_crop.shape

                    lower_half = vehicle_crop[h//2:h, :]


                    ocr_result = reader.readtext(lower_half)

                    plate_text = "UNKNOWN"

                    if len(ocr_result) > 0:

                        # Take longest detected text
                        texts = [r[1] for r in ocr_result]

                        plate_text = max(texts, key=len)


                    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

                    print(f"Vehicle {track_id} | Plate: {plate_text} | Time: {timestamp}")



                    records.append({
                        "Vehicle_ID": int(track_id),
                        "Plate_Number": plate_text,
                        "Timestamp": timestamp
                    })

    out.write(annotated_frame)


cap.release()
out.release()



df = pd.DataFrame(records)

df.to_csv("vehicle_records.csv", index=False)

print("FINAL PROCESS COMPLETE")

In [ ]:
from IPython.display import Video

Video("final_output.mp4", embed=True)

In [ ]:
from google.colab import files

files.download("vehicle_records.csv")

In [ ]:
from google.colab import files

files.download("final_output.mp4")